In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import pandas as pd
import time

from processing.load_datasets import load_rts
from processing.configs import COLORS_10

from sklearn.manifold import TSNE
from umap import UMAP
from trimap import TRIMAP
from pacmap import PaCMAP

In [ ]:
rts_df = load_rts(merge_metadata=True)

In [ ]:
sports_dict = {
    "Football": [
        "Football en direct",
        "Football enregistrement"
    ],
    "Tennis": [
        "Tennis en direct"
    ],
    "Alpine skiing and snowboarding": [
        "Ski alpin, snowboard en direct",
        "Ski alpin, snowboard enregistrement"
    ],
    "Cross country skiing": [
        "Ski de fond en direct"
    ],
    "Ice hockey": [
        "Hockey sur glace en direct"
    ],
    "Swimming": [
        "Natation en direct"
    ],
    "Volleyball": [
        "Volley-ball en direct"
    ],
    "Athletics": [
        "Athlétisme en direct",
        "Athlétisme enregistrement"
    ],
    "Cycling": [
        "Cyclisme en direct"
    ],
    "Motorcycling": [
        "Motocyclisme en direct"
    ]
}

In [ ]:
def map_sport_class(label, sports_dict, default=None):
    """
    Map a RTS sport class label to a canonical sport category.

    Parameters
    ----------
    label : str
        Original RTS class.
    sports_dict : dict
        Dictionary of canonical sport -> list of RTS labels.
    default : any
        Value returned if no match is found.

    Returns
    -------
    str or default
        Canonical sport category.
    """
    if label is None:
        return default

    for sport, labels in sports_dict.items():
        if label in labels:
            return sport

    return default

rts_df["sport"] = rts_df["contentType"].apply(lambda x: map_sport_class(x, sports_dict))

In [ ]:
rts_df.sport.value_counts()

In [ ]:
rts_df = rts_df.dropna(subset=["sport"])
print(f"Total samples after filtering: {len(rts_df)}")

In [ ]:
c = 100

dr_algos = {
    "tsne": TSNE(n_components=2, perplexity=50),
    "umap": UMAP(n_components=2, min_dist=0.5, n_neighbors=30),
    "trimap": TRIMAP(n_dims=2, n_inliers=2*c, n_outliers=c, n_random=c),
    "pacmap": PaCMAP(n_components=2, n_neighbors=30, MN_ratio=5.0, FP_ratio=5.0)
}

algo_names = list(dr_algos.keys())

In [ ]:
N_SAMPLE = 10000
rts_sample = rts_df.sample(N_SAMPLE, random_state=42)
X = np.stack(rts_sample["imagenet_features"].values)
y = rts_sample["contentType"].values

dr_results = {}
for algo_name, algo in dr_algos.items():
    output_path = f"embeddings/rts_sports/{algo_name}.npy"
    if os.path.exists(output_path):
        print(f"Loading existing embedding for {algo_name}...")
        X_dr = np.load(output_path)
        dr_results[algo_name] = {
            "embedding": X_dr,
            "time": None
        }
        continue

    print(f"Running {algo_name}...")
    start_time = time.time()
    X_dr = algo.fit_transform(X)
    end_time = time.time()
    dr_results[algo_name] = {
        "embedding": X_dr,
        "time": end_time - start_time
    }

    # Save embeddings
    np.save(output_path, X_dr)

    print(f"{algo_name} completed in {end_time - start_time:.2f} seconds")

In [ ]:
# Plotting the results colored by content type

sport_categories = list(sports_dict.keys())
color_map = {sport: COLORS_10[i] for i, sport in enumerate(sport_categories)}
color_list = [color_map[sport] for sport in rts_sample["sport"]]

padding_ratio = 0.1  # 5% padding around the embedding

titles = [
    "t-SNE: perplexity=50",
    "UMAP: min_dist=0.5, n_neighbors=30",
    "TriMap: n_inliers=2c, n_outliers=c, n_random=c (c=100)",
    "PaCMAP: n_neighbors=30, MN_ratio=5.0, FP_ratio=5.0"
]

fig, axes = plt.subplots(2, 2, figsize=(15, 15))

for i, algo_name in enumerate(algo_names):
    ax = axes[i // 2, i % 2]
    X_dr = dr_results[algo_name]["embedding"]

    ax.scatter(X_dr[:, 0], X_dr[:, 1], c=color_list, alpha=0.6, s=5)

    # --- Square + padding ---
    x_min, x_max = X_dr[:, 0].min(), X_dr[:, 0].max()
    y_min, y_max = X_dr[:, 1].min(), X_dr[:, 1].max()

    max_range = max(x_max - x_min, y_max - y_min)

    # Add padding
    padded_range = max_range * (1 + padding_ratio)

    x_center = (x_max + x_min) / 2
    y_center = (y_max + y_min) / 2

    ax.set_xlim(x_center - padded_range / 2, x_center + padded_range / 2)
    ax.set_ylim(y_center - padded_range / 2, y_center + padded_range / 2)

    ax.set_aspect("equal")

    ax.set_title(f"{titles[i]}", fontsize=16)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()

# Legend
handles = [mpatches.Patch(color=color_map[sport], label=sport) for sport in sport_categories]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0), ncol=5, frameon=False)

# Reserve space at bottom
plt.subplots_adjust(bottom=0.05)

plt.savefig("images/rts_sports_comparison.png", dpi=300, bbox_inches='tight')
plt.show()